# Tarea Nº1 — Inteligencia Artificial 2026-1
Universidad Diego Portales — Facultad de Ingeniería y Ciencias

**Estudiantes:** Maximiliano Oliva y Alonso Iturra

**Declaración de uso de herramientas generativas:** Gemini para complementar como a su vez revisión del código.

## Segunda parte
Para esta sección se empleará el dataset de Human Activity Recognition Using Smartphones. El cual es un dataset, en el cual participaron 30 sujetos, consiste en que cada sujeto con su telefono realiza una de seis actividades (Caminar, subir escaleras, bajar escaleras, sentarse, estar de pie, acostado).
Primero se uso la libreria pandas para poder importar los datos

In [27]:
# Ejecutar antes iniciar
!pip install pgmpy pandas numpy matplotlib

In [28]:
import pandas as pd
import numpy as np

# Definimos la ruta base donde esta el dataset descomprimido
path = 'UCI HAR Dataset/train/'

# Cargamos los archivos usando el separador de espacios en blanco '\s+'
# X_train: Los datos de los sensores (7352 filas, 561 columnas)
X_train = pd.read_csv(path + 'X_train.txt', sep=r'\s+', header=None)

# y_train: Las actividades/estados (7352 filas, 1 columna)
y_train = pd.read_csv(path + 'y_train.txt', header=None)

# subject_train: Quién es el dueño de cada fila (7352 filas, 1 columna)
subject_train = pd.read_csv(path + 'subject_train.txt', header=None)


Primero se definen los estados
$S_1: Caminar (Walking)$
$S_2​: Subir\ escaleras (Walking_Upstairs)$
$S_3​: Bajar\ escaleras (Walking_Downstairs)$
$S_4​: Sentado (Sitting)$
$S_5​: De\ pie (Standing)$
$S_6​: Acostado (Laying)$
Luego se crean los índices obteniendo niveles: bajo, medio y alto. Estas son para la aceleración y giroscopio, que en conjunto hacen 9 observaciones.
Se empleará la discretización mediante percentiles, usando los 33 y 66 para dividir los datos en 3 grandes grupos.

In [29]:
# CELDA CORREGIDA - Discretización
def discretizar(col):
    return pd.qcut(col, q=3, labels=[0,1,2])

Iacc = (X_train.iloc[:, 200] + X_train.iloc[:, 201]) / 2
Igyro = (X_train.iloc[:, 239] + X_train.iloc[:, 240]) / 2

# Primero discretizamos
Iacc_disc = discretizar(Iacc).astype(int)
Igyro_disc = discretizar(Igyro).astype(int)

# LUEGO creamos las observaciones
print("Observaciones")
observaciones = Iacc_disc * 3 + Igyro_disc  # ✅ Ahora está en el orden correcto

print("Discretización")
print(Iacc_disc.value_counts())
print(Igyro_disc.value_counts())

Observaciones
Discretización
0    2451
2    2451
1    2450
Name: count, dtype: int64
0    2451
2    2451
1    2450
Name: count, dtype: int64


PARTE B


In [30]:
data = pd.DataFrame({
    'subject': subject_train[0],
    'obs': observaciones,
    'state': y_train[0] - 1
})

secuencias_obs = []
secuencias_states = []

for subj in data['subject'].unique():
    df_subj = data[data['subject'] == subj]
    
    secuencias_obs.append(df_subj['obs'].values)
    secuencias_states.append(df_subj['state'].values)



In [31]:
n_states = 6
pi = np.zeros(n_states)

for seq in secuencias_states:
    pi[seq[0]] += 1

pi = pi / pi.sum()

In [32]:


A = np.zeros((n_states, n_states))

for seq in secuencias_states:
    for i in range(len(seq) - 1):
        A[seq[i], seq[i+1]] += 1

# Normalizar filas
A = A / A.sum(axis=1, keepdims=True)

In [33]:
n_obs = 9
B = np.zeros((n_states, n_obs))

for obs_seq, state_seq in zip(secuencias_obs, secuencias_states):
    for o, s in zip(obs_seq, state_seq):
        B[s, o] += 1

B = B / B.sum(axis=1, keepdims=True)

# 1. Imprimir Vector Inicial Pi
print("--- Vector de Probabilidades Iniciales (Pi) ---")
pi_df = pd.DataFrame(pi, index=[f"Estado {i+1}" for i in range(n_states)], columns=["Prob"])
print(pi_df)
print(f"\nSuma total de Pi: {pi.sum()}") # Validación

print("\n" + "="*50 + "\n")

# 2. Imprimir Matriz de Transición A
print("--- Matriz de Transición (A) ---")
# Etiquetas para entender qué fila pasa a qué columna
labels_estados = ["Caminar(1)", "Subir(2)", "Bajar(3)", "Sentado(4)", "Parado(5)", "Acostado(6)"]
A_df = pd.DataFrame(A, index=labels_estados, columns=labels_estados)
print(A_df.round(4)) # Redondeamos a 4 decimales para leer mejor
print(f"\nSuma de filas de A: {A.sum(axis=1)}") # Validación

print("\n" + "="*50 + "\n")

# 3. Imprimir Matriz de Emisión B
print("--- Matriz de Emisión (B) ---")
obs_labels = [f"Obs {i}" for i in range(n_obs)]
B_df = pd.DataFrame(B, index=labels_estados, columns=obs_labels)
print(B_df.round(4))
print(f"\nSuma de filas de B: {B.sum(axis=1)}") # Validación

--- Vector de Probabilidades Iniciales (Pi) ---
          Prob
Estado 1   0.0
Estado 2   0.0
Estado 3   0.0
Estado 4   0.0
Estado 5   1.0
Estado 6   0.0

Suma total de Pi: 1.0


--- Matriz de Transición (A) ---
             Caminar(1)  Subir(2)  Bajar(3)  Sentado(4)  Parado(5)  \
Caminar(1)       0.9657    0.0000    0.0343      0.0000     0.0000   
Subir(2)         0.0000    0.9668    0.0133      0.0000     0.0199   
Bajar(3)         0.0000    0.0549    0.9451      0.0000     0.0000   
Sentado(4)       0.0000    0.0000    0.0000      0.9666     0.0000   
Parado(5)        0.0000    0.0000    0.0000      0.0306     0.9694   
Acostado(6)      0.0299    0.0000    0.0000      0.0007     0.0000   

             Acostado(6)  
Caminar(1)        0.0000  
Subir(2)          0.0000  
Bajar(3)          0.0000  
Sentado(4)        0.0334  
Parado(5)         0.0000  
Acostado(6)       0.9694  

Suma de filas de A: [1. 1. 1. 1. 1. 1.]


--- Matriz de Emisión (B) ---
              Obs 0   Obs 1  Obs 2  

# HMM


In [34]:
from hmmlearn import hmm

modelo = hmm.CategoricalHMM(n_components=6, init_params="")
modelo.startprob_ = pi
modelo.transmat_ = A
modelo.emissionprob_ = B

seq = secuencias_obs[0].reshape(-1, 1)

logprob = modelo.score(seq)
print(logprob)

logprob, estados_viterbi = modelo.decode(seq, algorithm="viterbi")

logprob, posteriors = modelo.score_samples(seq)


print(modelo.startprob_.sum())        # 1
print(modelo.transmat_.sum(axis=1))   # filas = 1
print(modelo.emissionprob_.sum(axis=1)) # filas = 1

-276.1352065491369
1.0
[1. 1. 1. 1. 1. 1.]
[1. 1. 1. 1. 1. 1.]


# Forward-Backward
calcula la *probabilidad de estar en cada estado en cada instante de tiempo*, dado que conocemos toda la secuencia de observaciones. Calcula:

- *Forward ($\alpha$):* Probabilidad de observar $o_1, o_2, ..., o_t$ y estar en estado $i$ en el tiempo $t$.
- *Backward ($\beta$):* Probabilidad de observar $o_{t+1}, ..., o_T$ dado que estamos en estado $i$ en $t$.
- *Posterior ($\gamma$):* Combina ambos: $\gamma_t(i) = P(q_t = i | O, \lambda)$. Es la probabilidad a posteriori de estar en el estado $i$ en el tiempo $t$ dada *toda* la secuencia observada.

Se selecciona una subsecuencia real de longitud 20 del sujeto 1 para aplicar el algoritmo

In [35]:
seq = secuencias_obs[0][:20].reshape(-1, 1)
logprob, posteriors = modelo.score_samples(seq)
print(posteriors.shape)
print(posteriors.sum(axis=1))
print(posteriors[0])
estados_probables = np.argmax(posteriors, axis=1)

real = secuencias_states[0][:20]

print(estados_probables)
print(real)

(20, 6)
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[0. 0. 0. 0. 1. 0.]
[4 4 4 4 4 4 4 4 4 4 4 4 4 3 3 3 3 3 3 3]
[4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4]


# Viterbi
Encuentra la *secuencia de estados más probable* dada toda la secuencia de observaciones. A diferencia de Forward-Backward (que da probabilidades marginales por instante), Viterbi *optimiza la secuencia completa* de forma conjunta.

También calculamos *P(O|modelo)*: la probabilidad total de observar la secuencia bajo nuestro modelo, usando el algoritmo Forward.

In [36]:
seq = secuencias_obs[0][:20].reshape(-1, 1)
real = secuencias_states[0][:20]

logprob, estados_viterbi = modelo.decode(seq, algorithm="viterbi")

print(estados_viterbi)
print(real)

[4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4]
[4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4]


# Análisis
Como se mencionó anteriormente, se comenzó por discretizar las dos variables (aceleración y giroscopio) en tres niveles. Una vez hecho esto, se crearon las nueve observaciones producto de la combinación de dichos niveles, utilizando percentiles para categorizar cada valor. Por otro lado, se definieron los seis estados posibles: caminar, subir escaleras, bajar escaleras, sentarse, estar de pie y acostado.

## PI, Matriz emisión y transición
Con el entorno listo, se procedió a calcular el vector $\pi$, obteniendo que todas las secuencias parten del estado 5 (parado). Luego, se calculó la matriz de transición, obteniendo las probabilidades de pasar de un estado a otro; un punto importante a resaltar es que la suma de las filas es igual a 1, representando la probabilidad total. Por último, se calculó la matriz de emisión, donde se determina la probabilidad de pertenecer a un estado dada una observación. Por ejemplo: si se registra una aceleración y rotación (giroscopio) bajas, es probable que el sujeto se encuentre sentado, parado o acostado.

## Backward-Forward y Viterbi
Se inició con el algoritmo Forward-Backward, mediante el cual se estimó la probabilidad de una secuencia de longitud 20. El modelo determinó que la secuencia comienza en el estado 5; sin embargo, la secuencia calculada difiere de la real en los últimos siete estados. Esto puede deberse a que las observaciones del estado 3 son muy similares a las del estado 4 en términos de probabilidades de emisión, lo que genera confusión en el modelo al realizar la estimación local paso a paso.
Por otro lado, con el algoritmo de Viterbi, el modelo no falló, ya que estimó correctamente la secuencia de 20 pasos al considerar la probabilidad global de la trayectoria más probable.